# Step 3 · Model 1: Baseline (category lookup table)

**Goal:** give every campaign the success rate of its category, measured on training (2019–23) only.
This is the bar Models 2, 3 and 4 have to beat. A model that cannot beat "look up the category average"
has not learned anything worth shipping.

**What the model is.** One number per category, nothing else:

> `score(campaign) = average of the success column, over the training campaigns in that campaign's category`

No fitting, no settings to tune, no inputs other than the category label.

**What it produces.** `baseline_val.parquet` — a score between 0 and 1 for every 2024 campaign, so Models 2–4
can be compared against it on exactly the same rows.

**Results on validation (2024), filled in after running this notebook:**

| | |
|---|---|
| AUC (subcategory lookup) | **0.807** ← the number to beat |
| AUC (parent category lookup) | 0.737 |
| AUC (no model, everyone gets 73.5%) | 0.500 |
| Near-miss hit rate, riskiest 2,000 | 1.85% vs 2.02% random — **worse than random** |
| Near-miss hit rate, 20–40% score bracket | 3.76% vs 2.02% random — 1.9× |

⚠️ **The headline finding is the last two rows.** The baseline ranks success well (AUC 0.807) but the
campaigns it calls riskiest are *not* near-misses — they are campaigns that raise almost nothing.
Near-misses sit in the middle of the score range. Section 8 shows this, and it changes how Step 5 should
pick the coaching list for *every* model, not just this one.

**Team rules followed:** inputs are launch-day only (no `post_` columns), every rate is counted on
training (2019–23) only, and the 2025 test file is never opened.

### PIPELINE — Load the split files
Loads Step 2's output. Model 1 needs only three columns, so it reads the full `slice_` spreadsheets
rather than the 191-column `X_` files: the category label lives in `slice_`, and `X_` holds it only as
61 one-hot columns.

Checks: 119,297 training rows at 73.5% success, 26,739 validation rows at 72.2%.

In [1]:
import os
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix

# Colab: upload split.zip and unzip to /content/split
BASE = "/content/split"
if not os.path.isdir(BASE):            # running locally instead
    BASE = "."

train = pd.read_parquet(f"{BASE}/slice_train.parquet")   # 2019-2023
val   = pd.read_parquet(f"{BASE}/slice_val.parquet")     # 2024
# slice_test.parquet (2025) stays shut until the final run.

for name, s in [("train 2019-23", train), ("val   2024   ", val)]:
    print(f"{name}: {len(s):>7,} campaigns | "
          f"success {s['success'].mean():.1%} | "
          f"near-miss {s['near_miss'].sum():,} ({s['near_miss'].mean():.2%})")

train 2019-23: 119,297 campaigns | success 73.5% | near-miss 1,923 (1.61%)
val   2024   :  26,739 campaigns | success 72.2% | near-miss 541 (2.02%)


### PIPELINE — Build the lookup table (training only)
One line per model. `groupby(...).mean()` on the `success` column is exactly "the share of campaigns in
this category that got funded", counted on 2019–23 only so later years never shape the model.

Two candidate category columns exist, both built in Step 2:
- `parent_category` — 15 labels (Games, Technology, Comics …)
- `subcategory` — 61 labels (Games / Tabletop Games, Technology / Apps …), the top 60 plus "other"

The next cell builds both; section 3 picks between them.

In [2]:
GLOBAL_RATE = train["success"].mean()     # fallback for a category never seen in training

def build_lookup(column):
    """The whole of Model 1: one average per category, from training only."""
    return train.groupby(column)["success"].mean()

def apply_lookup(table, slice_df, column):
    """Give every campaign its category's training rate."""
    return slice_df[column].map(table).fillna(GLOBAL_RATE).to_numpy()

lookup_parent = build_lookup("parent_category")
lookup_sub    = build_lookup("subcategory")

print(f"global training success rate : {GLOBAL_RATE:.4f}")
print(f"parent_category lookup       : {len(lookup_parent)} rows")
print(f"subcategory lookup           : {len(lookup_sub)} rows")

global training success rate : 0.7350
parent_category lookup       : 15 rows
subcategory lookup           : 61 rows


### PIPELINE — Check every validation category was seen in training
If a 2024 campaign had a category missing from 2019–23, it would fall back to the global rate and the
lookup would be guessing. Step 2 learned the top-60 list from training and applied it to every slice, so
this should be 0 — this cell confirms it rather than assuming it.

Result: 0 unseen labels in both columns, so the `fillna` above never fires.

In [3]:
for col, table in [("parent_category", lookup_parent), ("subcategory", lookup_sub)]:
    unseen = set(val[col]) - set(table.index)
    covered = val[col].isin(table.index).mean()
    print(f"{col:16s}: {len(unseen)} unseen labels in 2024, {covered:.1%} of campaigns covered")

parent_category : 0 unseen labels in 2024, 100.0% of campaigns covered
subcategory     : 0 unseen labels in 2024, 100.0% of campaigns covered


### PIPELINE — Score the validation campaigns, and measure AUC
AUC answers: *pick one campaign that succeeded and one that failed — did the model score the successful
one higher?* AUC is the share of such pairs it gets right. 0.5 is a coin flip, 1.0 is perfect
(L6; HW4 Part 4).

A third score is included as a floor: give **every** campaign the same global rate. It is a legitimate
"model", and its AUC is 0.500 by construction, because a model that cannot tell any two campaigns apart
gets every pair wrong half the time.

In [4]:
score_const  = np.full(len(val), GLOBAL_RATE)                       # no information at all
score_parent = apply_lookup(lookup_parent, val, "parent_category")  # 15 categories
score_sub    = apply_lookup(lookup_sub,    val, "subcategory")      # 61 categories

y_val  = val["success"].to_numpy()
nm_val = val["near_miss"].to_numpy()

print(f"{'lookup column':<22}{'distinct scores':>16}{'AUC on 2024':>14}")
print("-" * 52)
for name, s in [("none (global rate)", score_const),
                ("parent_category",    score_parent),
                ("subcategory",        score_sub)]:
    print(f"{name:<22}{len(np.unique(s)):>16,}{roc_auc_score(y_val, s):>14.4f}")

lookup column          distinct scores   AUC on 2024
----------------------------------------------------


none (global rate)                   1        0.5000
parent_category                     15        0.7366
subcategory                         60        0.8067


### EXPLORATION — Which category column should Model 1 use?
`subcategory` wins clearly: **AUC 0.807 vs 0.737**. Finer categories carry real information — inside
Technology, "Apps" funds 9% of the time and "Technology" (the catch-all) nearly always, and the
15-label column averages that distinction away.

**Decision: Model 1 is the `subcategory` lookup.** Taking the stronger of the two keeps the bar honest —
Models 2–4 should have to beat the best simple thing, not the weakest.

This matters: 0.807 is a high bar from a model with no fitting at all. If Model 2 (logistic regression)
lands near it, the extra 190 input columns are earning very little, and that is a finding worth reporting
rather than hiding.

In [5]:
score_val = score_sub          # <-- Model 1, used from here on
AUC_MODEL1 = roc_auc_score(y_val, score_val)
print(f"Model 1 (subcategory lookup) AUC on 2024 = {AUC_MODEL1:.4f}")
print(f"gain over parent_category                = {AUC_MODEL1 - roc_auc_score(y_val, score_parent):+.4f}")
print(f"gain over no model                       = {AUC_MODEL1 - 0.5:+.4f}")

Model 1 (subcategory lookup) AUC on 2024 = 0.8067
gain over parent_category                = +0.0700
gain over no model                       = +0.3067


### EXPLORATION — The lookup table itself
The whole model, readable in one screen. The spread is enormous — 8.8% to 100% — which is why such a
simple rule reaches 0.807.

`rate_train` is the model. `rate_val` is what actually happened in 2024 and is shown only to see where
the rule holds up. Two patterns worth carrying into the write-up:
- The bottom of the table is stable: Technology / Apps predicted 8.8%, actual 9.2%.
- The top of the table is optimistic: several categories predicted above 99% land in the 74–92% range in
  2024. Section 10 looks at why.

In [6]:
table = (train.groupby("subcategory").agg(n_train=("success", "size"),
                                          rate_train=("success", "mean"))
         .join(val.groupby("subcategory").agg(n_val=("success", "size"),
                                              rate_val=("success", "mean")))
         .sort_values("rate_train"))

pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("LOWEST 12 — where coaching money would go\n")
print(table.head(12).to_string())
print("\nHIGHEST 12 — near-certain funding\n")
print(table.tail(12).to_string())
print(f"\nsmallest training cell: {table['n_train'].min():,} campaigns "
      f"(no category is thin enough to need smoothing — see section 9)")

LOWEST 12 — where coaching money would go

                          n_train  rate_train  n_val  rate_val
subcategory                                                   
Technology / Apps            3193       0.088    606     0.092
Technology / Web              978       0.107    187     0.096
Games / Mobile Games          732       0.115    111     0.126
Technology / Software         995       0.138    242     0.145
Food / Food Trucks            620       0.161     80     0.087
Music / Hip-Hop               619       0.210     90     0.289
Film & Video / Webseries      620       0.295     72     0.278
Food / Restaurants           1147       0.310    205     0.337
Crafts / DIY                  521       0.313    104     0.279
Food / Drinks                1196       0.360    179     0.263
Film & Video / Animation     1172       0.361    281     0.391
Fashion / Footwear            560       0.414     72     0.514

HIGHEST 12 — near-certain funding

                               n_train 

### EXPLORATION — Why a 0.50 cutoff is the wrong line to draw
The usual cutoff — "flag it if the predicted chance of success is below 50%" — is a poor fit here, for a
reason specific to this problem rather than to this model:

- It flags **3,240** of 26,739 campaigns. Nobody has decided Kickstarter can coach 3,240 campaigns, so
  0.50 is not a business decision, it is an accident of where the category averages happen to fall.
- The cutoff moves with the base rate, not with capacity. Training is 73.5% successful, so 0.50 sits far
  down the distribution and lands inside one crowded block of categories.

**Instead we cut by capacity.** The team decides how many campaigns coaches can call in a year, and the
model fills that many slots, riskiest first — the same shape as HW4's `replacement_plan`, where a fixed
crew size set the budget. `COACH_CAPACITY` below is a placeholder to be replaced with a real staffing
number; every metric downstream is reported as a function of it.

In [7]:
print(f"campaigns scored below 0.50 by the subcategory lookup: {(score_val < 0.50).sum():,}"
      f"  ({(score_val < 0.50).mean():.1%} of 2024)")
print(f"subcategories whose training rate is below 0.50      : {(lookup_sub < 0.50).sum()} of {len(lookup_sub)}")

campaigns scored below 0.50 by the subcategory lookup: 3,240  (12.1% of 2024)
subcategories whose training rate is below 0.50      : 19 of 61


### PIPELINE — The coaching list, and the three metrics
`coaching_list()` is this project's version of HW4's `replacement_plan()`: it takes any model's scores,
fills the coaching capacity riskiest-first, and reports what that list would have achieved in 2024.

**One honest caveat built into the function.** Model 1 produces only 60 distinct scores — 61 categories,
two of them tied at exactly 100% — so the capacity
cut lands *inside* a block of tied campaigns — at capacity 2,000, 1,876 campaigns score strictly below the
cut and 281 are tied at it, so 124 of the tied block are picked arbitrarily. Ties are broken by campaign
id so the list is reproducible. This is a genuine weakness of a lookup table, not a bug: it cannot rank
two Tabletop Games campaigns against each other, which is exactly the gap Models 2–4 exist to close.

In [8]:
COACH_CAPACITY = 2_000     # campaigns Kickstarter can coach in a year -- TEAM DECISION, placeholder
RANDOM_NM_RATE = nm_val.mean()

def coaching_list(score, label, capacity=COACH_CAPACITY, quiet=False):
    """Fill the coaching capacity riskiest-first; report what that list caught in 2024."""
    score = np.asarray(score, dtype=float)
    order = np.lexsort((val["id"].to_numpy(), score))   # lowest score first; id breaks ties
    called = order[:capacity]

    near_misses = int(nm_val[called].sum())
    failures    = int((1 - y_val[called]).sum())
    summary = {"model": label,
               "campaigns called": capacity,
               "near-misses caught": near_misses,
               "near-miss hit rate": near_misses / capacity,
               "lift vs random": (near_misses / capacity) / RANDOM_NM_RATE,
               "failures caught": failures,
               "share of all failures": failures / (1 - y_val).sum()}
    if not quiet:
        print(f"{label}")
        print(f"  campaigns called        : {capacity:,}")
        print(f"  near-misses caught      : {near_misses:,} of {int(nm_val.sum()):,} "
              f"({near_misses / nm_val.sum():.1%} of all near-misses)")
        print(f"  near-miss hit rate      : {near_misses / capacity:.2%}   "
              f"(random: {RANDOM_NM_RATE:.2%})")
        print(f"  lift vs calling at random: {(near_misses / capacity) / RANDOM_NM_RATE:.2f}x")
        print(f"  failures caught         : {failures:,} "
              f"({failures / (1 - y_val).sum():.1%} of all 2024 failures)")
    return summary

results_model1 = coaching_list(score_val, "Model 1: subcategory lookup")

Model 1: subcategory lookup
  campaigns called        : 2,000
  near-misses caught      : 37 of 541 (6.8% of all near-misses)
  near-miss hit rate      : 1.85%   (random: 2.02%)
  lift vs calling at random: 0.91x
  failures caught         : 1,632 (21.9% of all 2024 failures)


### PIPELINE — Confusion matrix at the coaching capacity
The four boxes from the plan, with "flagged" meaning *we spend a coach's time on this campaign*.
The positive class is **failure**, because that is what we are hunting.

At capacity 2,000 the list is 81.6% accurate at picking campaigns that failed — but it reaches only 21.9%
of 2024's failures, because 2,000 slots cannot cover 7,443 failures. Precision is high and recall is low
by design; the capacity, not the model, sets recall.

In [9]:
flagged = np.zeros(len(val), dtype=bool)
flagged[np.lexsort((val["id"].to_numpy(), score_val))[:COACH_CAPACITY]] = True
failed = (y_val == 0)

tn, fp, fn, tp = confusion_matrix(failed, flagged).ravel()
print(f"{'':>22}{'not flagged':>14}{'flagged':>12}")
print(f"{'actually succeeded':>22}{tn:>14,}{fp:>12,}")
print(f"{'actually failed':>22}{fn:>14,}{tp:>12,}")
print(f"\nof the {COACH_CAPACITY:,} campaigns we call, {tp / (tp + fp):.1%} did fail   (precision)")
print(f"of the {int(failed.sum()):,} campaigns that failed, we called {tp / (tp + fn):.1%}   (recall)")
print(f"\nrecall is capped by capacity: {COACH_CAPACITY:,} slots / {int(failed.sum()):,} failures "
      f"= {COACH_CAPACITY / failed.sum():.1%} is the most any model could reach here.")

                         not flagged     flagged
    actually succeeded        18,928         368
       actually failed         5,811       1,632

of the 2,000 campaigns we call, 81.6% did fail   (precision)
of the 7,443 campaigns that failed, we called 21.9%   (recall)

recall is capped by capacity: 2,000 slots / 7,443 failures = 26.9% is the most any model could reach here.


### EXPLORATION — ⚠️ Near-misses are not the riskiest campaigns
**This is the most important result in the notebook, and it is not about Model 1.**

The coaching list above has a near-miss hit rate of **1.85%** against a random rate of **2.02%**. Calling
the campaigns the model thinks are riskiest is *worse than calling 2,000 campaigns at random*.

The reason is visible in the table below. A campaign scored under 20% does fail — but it fails at 3% of
goal, not 80%. It is not a near-miss; it is a campaign that was never close. Near-misses cluster where
the model is *uncertain*: the 20–80% score range runs at 1.5–2.1× the random near-miss rate, while both
ends of the range run at 0.5–0.7×.

**What this means for the project:**
1. Ranking by predicted failure is the wrong coaching rule, for every model, not just this one. Step 5's
   bracket approach is the right instinct, and this is the evidence for it.
2. AUC on `success` stays the right metric for *choosing* a model, but it is not the business metric. A
   model can improve AUC and still build a worse coaching list.
3. Worth raising with the team: the models predict `success`, and the business cares about `near_miss`.
   Those are different targets. Keeping `success` is defensible — near-misses are only 2% of campaigns,
   so a direct model would train on very few positives — but the plan should say *why*, not leave it
   implicit.

In [10]:
bins = np.arange(0, 1.01, 0.1)
d = pd.DataFrame({"score": score_val, "success": y_val, "near_miss": nm_val})
d["failed"] = 1 - d["success"]
d["band"] = pd.cut(d["score"], bins, include_lowest=True)

by_band = d.groupby("band", observed=False).agg(
    campaigns=("success", "size"),
    actual_success=("success", "mean"),
    failures=("failed", "sum"),
    near_misses=("near_miss", "sum"),
    near_miss_rate=("near_miss", "mean"))
by_band["lift_vs_random"] = by_band["near_miss_rate"] / RANDOM_NM_RATE
by_band["nm_share_of_failures"] = by_band["near_misses"] / by_band["failures"].replace(0, np.nan)

print(f"random near-miss rate in 2024 = {RANDOM_NM_RATE:.2%}\n")
print(by_band.to_string())
print("\nlift < 1 at both ends, lift > 1.5 in the middle: the near-misses are in the middle.")

random near-miss rate in 2024 = 2.02%

               campaigns  actual_success  failures  near_misses  near_miss_rate  lift_vs_random  nm_share_of_failures
band                                                                                                                 
(-0.001, 0.1]        606           0.092       550            8           0.013           0.652                 0.015
(0.1, 0.2]           620           0.119       546            7           0.011           0.558                 0.013
(0.2, 0.3]           162           0.284       116            6           0.037           1.831                 0.052
(0.3, 0.4]           769           0.332       514           29           0.038           1.864                 0.056
(0.4, 0.5]          1083           0.485       558           47           0.043           2.145                 0.084
(0.5, 0.6]          6018           0.522      2876          193           0.032           1.585                 0.067
(0.6, 0.7]       

### EXPLORATION — Coaching rules compared (Step 5 preview)
Same scores, same 2024 campaigns, different rules for who gets called. Riskiest-first loses to random;
every middle bracket beats it.

The 20–40% bracket is the best rule available from Model 1: **931 campaigns at a 3.76% hit rate, 1.9×
random**. Note that it is a *list of 931*, not a list of 2,000 — a lookup table with 61 values cannot
produce a bracket of any chosen size. Giving Step 5 brackets that match the real coaching capacity is
another job for Models 2–4.

In [11]:
def rule(label, mask):
    n = int(mask.sum())
    hits = int(nm_val[mask].sum())
    hit_rate = hits / n if n else 0.0
    print(f"  {label:<36}{n:>8,}{hits:>12,}{hit_rate:>11.2%}{hit_rate / RANDOM_NM_RATE:>9.2f}x")

print(f"  {'coaching rule':<36}{'called':>8}{'near-miss':>12}{'hit rate':>11}{'lift':>9}")
print("  " + "-" * 76)
rule("call everyone (= random)", np.ones(len(val), bool))
rule(f"riskiest {COACH_CAPACITY:,} (lowest score)", flagged)
print("  " + "-" * 76)
edges = [0, .2, .4, .6, .8, 1.01]
for lo, hi in zip(edges[:-1], edges[1:]):
    rule(f"score bracket {lo:.0%}-{min(hi,1):.0%}", (score_val >= lo) & (score_val < hi))

  coaching rule                         called   near-miss   hit rate     lift


  ----------------------------------------------------------------------------
  call everyone (= random)              26,739         541      2.02%     1.00x
  riskiest 2,000 (lowest score)          2,000          37      1.85%     0.91x
  ----------------------------------------------------------------------------
  score bracket 0%-20%                   1,226          15      1.22%     0.60x
  score bracket 20%-40%                    931          35      3.76%     1.86x
  score bracket 40%-60%                  7,101         240      3.38%     1.67x
  score bracket 60%-80%                  1,189          39      3.28%     1.62x
  score bracket 80%-100%                16,292         212      1.30%     0.64x


### EXPLORATION — Does smoothing the lookup help? No
A standard worry with lookup tables: a category with few campaigns gets a noisy average, and categories
at exactly 0% or 100% are almost certainly overconfident. The usual fix shrinks each category toward the
global rate, by `k` imaginary average campaigns.

Tested across `k = 0` to `250`: AUC moves from 0.8067 to at best 0.8076 at `k = 50`, a gain of 0.001.
**Model 1 stays unsmoothed.** Step 2's top-60 rule did the work already — the smallest category still
holds 495 training campaigns, so there is no thin cell left to rescue, and a baseline should stay the
simplest defensible thing.

In [12]:
cells_ = train.groupby("subcategory")["success"].agg(["size", "mean"])
print(f"{'k (shrinkage)':>14}{'AUC on 2024':>14}{'lowest score':>15}{'highest score':>15}")
print("-" * 58)
for k in [0, 5, 25, 50, 100, 250]:
    smoothed = (cells_["mean"] * cells_["size"] + GLOBAL_RATE * k) / (cells_["size"] + k)
    s = apply_lookup(smoothed, val, "subcategory")
    print(f"{k:>14}{roc_auc_score(y_val, s):>14.4f}{smoothed.min():>15.3f}{smoothed.max():>15.3f}")

 k (shrinkage)   AUC on 2024   lowest score  highest score
----------------------------------------------------------


             0        0.8067          0.088          1.000
             5        0.8067          0.089          0.999


            25        0.8073          0.093          0.998
            50        0.8076          0.098          0.996
           100        0.8063          0.107          0.993
           250        0.8053          0.135          0.990


### EXPLORATION — Data-quality flag for the team: the "X / X" categories
Seven categories repeat their parent's name — `Art / Art`, `Music / Music`, `Technology / Technology` and
so on. These are campaigns whose JSON had no sub-level, so Step 2's label-building fell back to the
parent name.

They behave oddly: **8,915 training campaigns (7.5%) at a 99.8% average success rate**, dropping to 89.4%
in 2024 — `Music / Music` falls from 99.5% to 74.1%. A near-perfect training rate that does not hold up
is the signature of a cell that is not a real category.

Not blocking for Model 1 — the effect on AUC is small and these campaigns are scored at the top of the
range where few near-misses live. Flagged here because Models 2–4 use the same 61 one-hot columns and
will inherit it, and because it belongs in the README's limitations.

In [13]:
same_name = [c for c in table.index
             if " / " in c and c.split(" / ")[0] == c.split(" / ")[1]]
flagged_cells = table.loc[same_name].sort_values("n_train", ascending=False)
print(flagged_cells.to_string())
print(f"\n{flagged_cells['n_train'].sum():,} training campaigns "
      f"({flagged_cells['n_train'].sum() / len(train):.1%} of training)")
print(f"average predicted {flagged_cells['rate_train'].mean():.1%} -> "
      f"actual {flagged_cells['rate_val'].mean():.1%} in 2024")

                         n_train  rate_train  n_val  rate_val
subcategory                                                  
Art / Art                   3988       0.999    125     0.920
Music / Music               1519       0.995     81     0.741
Comics / Comics              786       1.000     46     0.957
Publishing / Publishing      747       0.997     35     0.943
Design / Design              719       0.999     41     0.976
Technology / Technology      605       1.000     40     0.925
Crafts / Crafts              551       0.995     34     0.794

8,915 training campaigns (7.5% of training)
average predicted 99.8% -> actual 89.4% in 2024


### EXPLORATION — Calibration: the scores run high
Separate from ranking. The lookup predicts 77.9% of 2024 campaigns will succeed; 72.2% did — about
**5.7 points optimistic**, because it learned on 2019–23 when success averaged 73.5% and 2024 was a
slightly harder year.

This does not affect AUC (adding a constant to every score changes no ranking) but it does affect Step 5,
where the brackets are read as real probabilities. Noting it now so Models 2–4 get checked the same way.

In [14]:
print(f"mean predicted success, 2024 : {score_val.mean():.1%}")
print(f"actual success, 2024         : {y_val.mean():.1%}")
print(f"optimism                     : {score_val.mean() - y_val.mean():+.1%}")
print(f"\ntraining success rate 2019-23: {GLOBAL_RATE:.1%}  (what the lookup absorbed)")

mean predicted success, 2024 : 77.9%
actual success, 2024         : 72.2%
optimism                     : +5.7%

training success rate 2019-23: 73.5%  (what the lookup absorbed)


### PIPELINE — Save Model 1's outputs
Two files, so Models 2–4 compare against identical rows and nobody rebuilds the baseline:
- `baseline_val.parquet` — one row per 2024 campaign: `id`, `score_model1`, `success`, `near_miss`
- `baseline_lookup.csv` — the 61-row lookup table, the model itself

The lookup table is saved rather than recomputed because it is also what gets shown on a slide: "Apps
fund 9% of the time, Tabletop Games 99.5%" is the whole model, and it is readable by a non-technical
stakeholder.

In [15]:
out = pd.DataFrame({"id": val["id"].to_numpy(),
                    "score_model1": score_val,
                    "success": y_val,
                    "near_miss": nm_val})
out.to_parquet(f"{BASE}/baseline_val.parquet", index=False)
lookup_sub.rename("success_rate_train").to_csv(f"{BASE}/baseline_lookup.csv")

print(f"saved baseline_val.parquet   {out.shape[0]:,} rows x {out.shape[1]} columns")
print(f"saved baseline_lookup.csv    {len(lookup_sub)} categories")
print()
print(out.head().to_string(index=False))

saved baseline_val.parquet   26,739 rows x 4 columns
saved baseline_lookup.csv    61 categories



        id  score_model1  success  near_miss
1702314537         0.310        1          0
 853086300         0.995        1          0
 129968201         0.995        1          0
1171697795         0.999        1          0
 764915614         0.995        1          0


### Results — Model 1 summary

| Metric (validation 2024) | Model 1 | Reference |
|---|---|---|
| AUC | **0.807** | 0.500 = coin flip; 0.737 = parent category |
| Precision of the 2,000-campaign coaching list | 81.6% | 27.8% if called at random |
| Share of 2024 failures reached | 21.9% | capped at 26.9% by capacity |
| Near-miss hit rate, riskiest-first | 1.85% | 2.02% random — **loses** |
| Near-miss hit rate, 20–40% bracket | 3.76% | 2.02% random — 1.9× |
| Calibration | +5.7 points optimistic | — |

**What Models 2–4 have to beat: AUC 0.807 on 2024.**

**Three things this baseline establishes for the rest of Step 3:**

1. **The bar is high, and that is informative.** Category alone reaches 0.807. If logistic regression with
   191 columns lands at 0.82, the honest headline is that category explains most of what is knowable on
   launch day — a better finding than a model with an unexplained number next to it.
2. **The baseline cannot rank inside a category**, which is the specific gap Models 2–4 fill. Goal size,
   campaign length and prep time all vary enormously inside "Games / Tabletop Games" and the lookup gives
   all of them 99.5%. Ties also make capacity-sized coaching lists impossible to cut cleanly.
3. **The coaching rule needs rethinking before Step 5**, for every model: riskiest-first loses to random
   on the metric Kickstarter actually cares about.

**Limitations.** 60 distinct scores and heavy ties; +5.7 points optimistic on 2024; inherits the
`X / X` category artifact; `other` lumps 18.9% of training into one cell, and in 2024 it holds 3,937
campaigns at a 2.9% near-miss rate, so real signal is hidden inside it.